# VisionAssist Phase 12 — Release readiness

This notebook performs no training. It verifies immutable promoted and rollback artifacts, pins the Qwen base and processor revision, runs a task-balanced 96-record clean-runtime acceptance assessment, requires a `ready` report, and only then builds the release bundle.

In [ ]:
#@title 1. Settings — run before importing Torch
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PROMOTED_RUN_ID = "qwen25vl3b_qlora_balanced_replay_v1"
ROLLBACK_RUN_ID = "qwen25vl3b_qlora_pilot_v1"
PINNED_REVISION = "66285546d2b821cf421d4f5eb2576359d3770cd3"

In [ ]:
#@title 2. Mount Drive and verify required external artifacts
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROMOTED = DRIVE_ROOT / "outputs/training" / PROMOTED_RUN_ID
DRIVE_ROLLBACK = DRIVE_ROOT / "outputs/training" / ROLLBACK_RUN_ID
required = [DRIVE_DATA_ARCHIVE, DRIVE_PROMOTED / "final_adapter/adapter_model.safetensors", DRIVE_ROLLBACK / "final_adapter/adapter_model.safetensors"]
for path in required: print(path, path.exists())
assert all(path.exists() for path in required)

In [ ]:
#@title 3. Clone/update repository and install dependencies
import shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["uv", "sync", "--extra", "training", "--extra", "dev"], cwd=PROJECT_ROOT, check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

In [ ]:
#@title 4. Verify GPU and pinned upstream revision
import torch
from huggingface_hub import model_info
assert torch.cuda.is_available(), "Select a GPU runtime."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0), round(properties.total_memory / 2**30, 2), "GiB")
resolved_revision = model_info("Qwen/Qwen2.5-VL-3B-Instruct", revision=PINNED_REVISION).sha
print("Resolved revision:", resolved_revision)
assert resolved_revision == PINNED_REVISION

## A. Restore and verify immutable inputs

In [ ]:
#@title 5. Restore prepared data and both adapters
import tarfile
data_files = [PROJECT_ROOT / "data/raw/visa", PROJECT_ROOT / "data/processed/visa_instructions/test.jsonl"]
if not all(path.exists() for path in data_files):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive: archive.extractall(PROJECT_ROOT)
for run_id, source in ((PROMOTED_RUN_ID, DRIVE_PROMOTED), (ROLLBACK_RUN_ID, DRIVE_ROLLBACK)):
    destination = PROJECT_ROOT / "outputs/training" / run_id
    shutil.copytree(source, destination, dirs_exist_ok=True)
    assert (destination / "final_adapter/adapter_model.safetensors").is_file()
print("Prepared data and immutable adapters restored.")

In [ ]:
#@title 6. Normalize portable image paths and verify frozen benchmark
import hashlib, json
from pathlib import PurePosixPath
MARKER = ("data", "raw", "visa")
def portable(value):
    parts = PurePosixPath(value.replace("\\", "/")).parts
    lowered = tuple(part.lower() for part in parts)
    for index in range(len(parts) - len(MARKER) + 1):
        if lowered[index:index + len(MARKER)] == MARKER: return PurePosixPath(*parts[index:]).as_posix()
    candidate = PurePosixPath(*parts)
    if not candidate.is_absolute() and ".." not in candidate.parts: return candidate.as_posix()
    raise ValueError(value)
for split in ("train", "validation", "test"):
    path = PROJECT_ROOT / f"data/processed/visa_instructions/{split}.jsonl"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    for row in rows:
        for message in row["messages"]:
            for item in message.get("content", []):
                if item.get("type") == "image": item["image"] = portable(item["image"])
    path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
benchmark = PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"
actual = hashlib.sha256(benchmark.read_bytes()).hexdigest()
expected = json.loads((benchmark.parent / "benchmark_manifest.json").read_text(encoding="utf-8"))["benchmark_sha256"]
print("Frozen benchmark SHA-256:", actual)
assert actual == expected

In [ ]:
#@title 7. Run Phase 12 CPU regression tests
subprocess.run(["uv", "run", "pytest", "tests/test_phase7c_inference.py", "tests/test_phase8_training.py", "tests/test_phase12_release.py"], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 8. Run pre-acceptance release verification
RELEASE_CONFIG = PROJECT_ROOT / "configs/release/phase12.yaml"
subprocess.run(["uv", "run", "visionassist", "release-readiness", "--config", str(RELEASE_CONFIG)], cwd=PROJECT_ROOT, check=True)
preflight = json.loads((PROJECT_ROOT / "outputs/release/phase12/release_readiness.json").read_text(encoding="utf-8"))
print(json.dumps(preflight, indent=2))
assert preflight["status"] == "pending"
assert preflight["counts"]["fail"] == 0

## B. Clean-runtime acceptance gate

In [ ]:
#@title 9. Configure persistence and audit the exact 96-record suite
import yaml
from collections import Counter
from visionassist.inference.generate import _load_benchmark
from visionassist.inference.schemas import load_inference_config
ACCEPTANCE_CONFIG = PROJECT_ROOT / "configs/inference/qwen25vl3b_phase12_acceptance.yaml"
payload = yaml.safe_load(ACCEPTANCE_CONFIG.read_text(encoding="utf-8"))
payload["persistent_output_dir"] = str(DRIVE_ROOT / "inference/qwen25vl3b_phase12_acceptance_v1")
ACCEPTANCE_CONFIG.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
acceptance_config = load_inference_config(ACCEPTANCE_CONFIG)
records = _load_benchmark(acceptance_config)
counts = Counter(record.task_family for record in records)
fingerprint = hashlib.sha256(("\n".join(record.instruction_id for record in records) + "\n").encode()).hexdigest()
print("records:", len(records), "tasks:", dict(sorted(counts.items())))
print("instruction fingerprint:", fingerprint)
assert len(records) == 96 and dict(counts) == acceptance_config.subset_task_quotas
assert fingerprint == "afb6725901b21a7f013bb082f644b202ec6856fb8f3895a465331737c537b2ef"

### Terminal progress monitor

While Cell 10 runs, open Colab **Terminal** and run:

```bash
watch -n 20 'date; echo LOCAL; wc -l /content/visionassist-industrial-visual-inspection/outputs/release/phase12/acceptance/predictions.partial.jsonl 2>/dev/null || true; echo DRIVE; wc -l /content/drive/MyDrive/visionassist/inference/qwen25vl3b_phase12_acceptance_v1/predictions.partial.jsonl 2>/dev/null || true; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader'
```

In [ ]:
#@title 10. Run task-balanced clean-runtime acceptance
RUN_ACCEPTANCE = False
if not RUN_ACCEPTANCE: raise RuntimeError("Acceptance gate is closed. Review Cells 4–9 first.")
subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(ACCEPTANCE_CONFIG)], cwd=PROJECT_ROOT, check=True)
torch.cuda.empty_cache()

In [ ]:
#@title 11. Require final ready status
subprocess.run(["uv", "run", "visionassist", "release-readiness", "--config", str(RELEASE_CONFIG), "--require-ready"], cwd=PROJECT_ROOT, check=True)
report_path = PROJECT_ROOT / "outputs/release/phase12/release_readiness.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))
assert report["status"] == "ready"
drive_acceptance = DRIVE_ROOT / "outputs/release/phase12/acceptance"
shutil.copytree(PROJECT_ROOT / "outputs/release/phase12/acceptance", drive_acceptance, dirs_exist_ok=True)
drive_report = DRIVE_ROOT / "outputs/release/phase12/release_readiness.json"
drive_report.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(report_path, drive_report)
print("Saved acceptance and ready report to Drive.")

## C. Explicit packaging gate

In [ ]:
#@title 12. Build and persist immutable release bundle
BUILD_RELEASE_BUNDLE = False
if not BUILD_RELEASE_BUNDLE: raise RuntimeError("Packaging gate is closed. Require ready status first.")
bundle = PROJECT_ROOT / "outputs/release/phase12/visionassist_phase12_v1"
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(PROJECT_ROOT / f"outputs/training/{PROMOTED_RUN_ID}/final_adapter", bundle / "promoted_adapter")
shutil.copytree(PROJECT_ROOT / f"outputs/training/{ROLLBACK_RUN_ID}/final_adapter", bundle / "rollback_adapter")
for source in (PROJECT_ROOT / "MODEL_CARD.md", PROJECT_ROOT / "docs/PHASE12_ROLLBACK.md", RELEASE_CONFIG, report_path):
    shutil.copy2(source, bundle / source.name)
acceptance_evidence = bundle / "acceptance_evidence"
acceptance_evidence.mkdir()
for source in (PROJECT_ROOT / "outputs/release/phase12/acceptance/run_manifest.json", PROJECT_ROOT / "outputs/release/phase12/acceptance/assessment_summary.json", PROJECT_ROOT / "outputs/release/phase12/acceptance/evaluation/metrics.json"):
    shutil.copy2(source, acceptance_evidence / source.name)
files = {}
for path in sorted(item for item in bundle.rglob("*") if item.is_file()): files[path.relative_to(bundle).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
manifest = {"schema_version": "1.0", "release_id": "visionassist_qwen25vl3b_phase12_v1", "model_revision": PINNED_REVISION, "files": files}
(bundle / "bundle_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
drive_release = DRIVE_ROOT / "releases/visionassist_phase12_v1"
shutil.copytree(bundle, drive_release, dirs_exist_ok=True)
archive = shutil.make_archive(str(DRIVE_ROOT / "releases/visionassist_phase12_v1"), "zip", root_dir=bundle.parent, base_dir=bundle.name)
print("Bundle:", drive_release)
print("Archive:", archive, "SHA-256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())

## Complete

Before disconnecting, confirm that Drive contains the acceptance directory, the final `ready` report, the expanded release directory, and the release ZIP. Share the final report and archive SHA-256 for repository evidence.